In [ ]:
!pip install langchain-huggingface sentence-transformers
!pip install langchain-huggingface langchain-chroma langchain-community faiss-cpu
!pip install -U langchain-classic

# 임베딩 모델(HuggingFace)

In [ ]:
# 설치 확인 후 임포트
from langchain_huggingface import HuggingFaceEmbeddings

# 1. 모델 로드
embeddings = HuggingFaceEmbeddings(
    model_name="intfloat/multilingual-e5-small",
    model_kwargs={"device": "cpu"},
    encode_kwargs={"normalize_embeddings": True},)

modules.json:   0%|          | 0.00/387 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/498k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/57.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/655 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  471MB            

...

sentencepiece.bpe.model: downloading bytes:           |  0.00B            

tokenizer.json: reconstructing file:   0%|          |  0.00B / 17.1MB            

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/167 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/200 [00:00<?, ?B/s]

In [ ]:
text = "임베딩 테스트를 하기 위한 샘플 문장입니다."

# 2. 쿼리 임베딩
query_result = embeddings.embed_query(text)
print(query_result[:5])

[0.04137963429093361, -0.05082991346716881, -0.034671083092689514, -0.06922543793916702, 0.09494147449731827]


In [ ]:
# 3. Document 임베딩
doc_result = embeddings.embed_documents([text])
print(doc_result[:5])

[[0.04137963429093361, -0.05082991346716881, -0.034671083092689514, -0.06922543793916702, 0.09494147449731827, -0.040435995906591415, 0.06483466178178787, 0.03947194665670395, 0.04323144257068634, 0.03378754109144211, 0.026907416060566902, 0.020828019827604294, 0.0893855020403862, -0.02423090860247612, 0.00014180902508087456, 0.034088678658008575, 0.04178820177912712, -0.05142945423722267, -0.09221949428319931, -0.030795972794294357, 0.009512069635093212, -0.010780426673591137, -0.03607817366719246, 0.06487013399600983, 0.030530286952853203, 0.04273645579814911, -0.05249964818358421, 0.02181941270828247, 0.05130547657608986, -0.04431132599711418, -0.03788643702864647, -0.03476405888795853, 0.01127375103533268, -0.059435177594423294, 0.034774329513311386, 0.013076644390821457, -0.020699886605143547, -0.03260912746191025, 0.07626847177743912, -0.09682678431272507, 0.02384602278470993, -0.008689092472195625, 0.06970789283514023, 0.06619533896446228, 0.036298755556344986, 0.046005483716726

In [ ]:
# 4. 차원 확인
print(len(doc_result[0]))  # 384 (모델 구조로 고정된 값)

384


In [ ]:
# 4. 유사도 계산
from sklearn.metrics.pairwise import cosine_similarity

sentence1 = "안녕하세요? 반갑습니다."
sentence2 = "안녕하세요? 반갑습니다!"
sentence3 = "안녕하세요? 만나서 반가워요."
sentence4 = "Hi, nice to meet you."
sentence5 = "I like to eat apples."

sentences = [sentence1, sentence2, sentence3, sentence4, sentence5]
embedded_sentences = embeddings.embed_documents(sentences)

def similarity(a, b):
    return cosine_similarity([a], [b])[0][0]

for i, s in enumerate(embedded_sentences):
    for j, o in enumerate(embedded_sentences):
        if i < j:
            print(f"[유사도 {similarity(s, o):.4f}] {sentences[i]} <=====> {sentences[j]}")

[유사도 0.9910] 안녕하세요? 반갑습니다. <=====> 안녕하세요? 반갑습니다!
[유사도 0.9451] 안녕하세요? 반갑습니다. <=====> 안녕하세요? 만나서 반가워요.
[유사도 0.8597] 안녕하세요? 반갑습니다. <=====> Hi, nice to meet you.
[유사도 0.7890] 안녕하세요? 반갑습니다. <=====> I like to eat apples.
[유사도 0.9339] 안녕하세요? 반갑습니다! <=====> 안녕하세요? 만나서 반가워요.
[유사도 0.8483] 안녕하세요? 반갑습니다! <=====> Hi, nice to meet you.
[유사도 0.7743] 안녕하세요? 반갑습니다! <=====> I like to eat apples.
[유사도 0.8670] 안녕하세요? 만나서 반가워요. <=====> Hi, nice to meet you.
[유사도 0.7840] 안녕하세요? 만나서 반가워요. <=====> I like to eat apples.
[유사도 0.8031] Hi, nice to meet you. <=====> I like to eat apples.


# Vector Store (+ 문서 임베딩)

In [ ]:
# 문서 업로드
!wget -P data/ https://raw.githubusercontent.com/teddylee777/langchain-kr/refs/heads/main/09-VectorStore/data/nlp-keywords.txt
!wget -P data/ https://raw.githubusercontent.com/teddylee777/langchain-kr/refs/heads/main/09-VectorStore/data/finance-keywords.txt

--2026-09-14 11:18:19--  https://raw.githubusercontent.com/teddylee777/langchain-kr/refs/heads/main/09-VectorStore/data/nlp-keywords.txt
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 12533 (12K) [text/plain]
Saving to: ‘data/nlp-keywords.txt’

nlp-keywords.txt    100%[===================>]  12.24K  --.-KB/s    in 0s      

2026-09-14 11:18:19 (99.7 MB/s) - ‘data/nlp-keywords.txt’ saved [12533/12533]

--2026-09-14 11:18:19--  https://raw.githubusercontent.com/teddylee777/langchain-kr/refs/heads/main/09-VectorStore/data/finance-keywords.txt
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connect

In [ ]:
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import CharacterTextSplitter

text_splitter = CharacterTextSplitter(chunk_size=600, chunk_overlap=0)

loader1 = TextLoader("data/nlp-keywords.txt")
loader2 = TextLoader("data/finance-keywords.txt")

split_doc1 = loader1.load_and_split(text_splitter)
split_doc2 = loader2.load_and_split(text_splitter)

# print(f"NLP 문서 청크 수: {len(split_doc1)}")
# print(f"Finance 문서 청크 수: {len(split_doc2)}")

/tmp/ipykernel_881/1113959915.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import TextLoader


### Chroma

In [ ]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma
from langchain_core.documents import Document
import shutil

embeddings = HuggingFaceEmbeddings(
    model_name="intfloat/multilingual-e5-small",
    model_kwargs={"device": "cpu"},
    encode_kwargs={"normalize_embeddings": True},
)

# 기존 저장소가 있으면 삭제하고 새로 시작 (중복 저장 방지)
shutil.rmtree("./chroma_db", ignore_errors=True)

# 1. 초기화 (벡터저장소 생성) - 문서만 넘기면 끝, persist_directory만 지정하면 자동 저장됨
db = Chroma.from_documents(
    documents=split_doc1 + split_doc2,
    embedding=embeddings,
    persist_directory="./chroma_db",)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

In [ ]:
# 중복 없이 저장됐는지 확인
stored = db.get()
print(len(stored["ids"]))

print(f"저장된 문서 수: {len(stored['ids'])}")
print(f"원본 청크 수: {len(split_doc1) + len(split_doc2)}")

17
저장된 문서 수: 17
원본 청크 수: 17


In [ ]:
# 2. 유사도 검색 (점수 포함)
query = "딥러닝에서 사용하는 핵심 개념은?"
results = db.similarity_search_with_score(query, k=2)
for doc, score in results:
    print(f"[거리 {score:.4f}] {doc.page_content[:50]}...")

[거리 0.2713] 정의: TF-IDF는 문서 내에서 단어의 중요도를 평가하는 데 사용되는 통계적 척도입니다....
[거리 0.2815] 정의: DataFrame은 행과 열로 이루어진 테이블 형태의 데이터 구조로, 주로 데이터 ...


In [ ]:
# 3. 텍스트로부터 추가 - metadatas를 명시적으로 넣기
db.add_texts(
    ["LangChain은 LLM 애플리케이션 개발을 돕는 프레임워크입니다."])

# 4. 문서(Document)로부터 추가
db.add_documents([Document(page_content="RAG는 검색 증강 생성 기법입니다.", metadata={"source": "추가문서"})])

['118da160-c6be-4abf-b289-a1d76910dd8a']

In [ ]:
# 5. 문서 삭제 (source가 "추가문서"인 것만 삭제)
stored = db.get()
target_ids = [i for i, m in zip(stored["ids"], stored["metadatas"]) if (m or {}).get("source") == "추가문서"]
db.delete(ids=target_ids)

In [ ]:
# 6. 저장 - persist_directory를 지정한 시점부터 자동 저장되므로 별도 호출 불필요

# 7. 저장된 데이터 확인
stored_data = db.get()
print(f"저장된 청크 수: {len(stored_data['ids'])}")
print(stored_data["documents"][0][:50])

저장된 청크 수: 18
Semantic Search

정의: 의미론적 검색은 사용자의 질의를 단순한 키워드 매칭을


In [ ]:
# 8. 검색기로 변환
retriever = db.as_retriever(search_type="similarity", search_kwargs={"k": 2})
retriever.invoke(query)

[Document(id='acd1d539-2ab3-4843-8906-b5151d38b2d7', metadata={'source': 'data/nlp-keywords.txt'}, page_content='정의: TF-IDF는 문서 내에서 단어의 중요도를 평가하는 데 사용되는 통계적 척도입니다. 이는 문서 내 단어의 빈도와 전체 문서 집합에서 그 단어의 희소성을 고려합니다.\n예시: 많은 문서에서 자주 등장하지 않는 단어는 높은 TF-IDF 값을 가집니다.\n연관키워드: 자연어 처리, 정보 검색, 데이터 마이닝\n\nDeep Learning\n\n정의: 딥러닝은 인공신경망을 이용하여 복잡한 문제를 해결하는 머신러닝의 한 분야입니다. 이는 데이터에서 고수준의 표현을 학습하는 데 중점을 둡니다.\n예시: 이미지 인식, 음성 인식, 자연어 처리 등에서 딥러닝 모델이 활용됩니다.\n연관키워드: 인공신경망, 머신러닝, 데이터 분석\n\nSchema\n\n정의: 스키마는 데이터베이스나 파일의 구조를 정의하는 것으로, 데이터가 어떻게 저장되고 조직되는지에 대한 청사진을 제공합니다.\n예시: 관계형 데이터베이스의 테이블 스키마는 열 이름, 데이터 타입, 키 제약 조건 등을 정의합니다.\n연관키워드: 데이터베이스, 데이터 모델링, 데이터 관리\n\nDataFrame'),
 Document(id='3fd325ee-306c-4170-af97-bd5a1fba3f9e', metadata={'source': 'data/nlp-keywords.txt'}, page_content="정의: DataFrame은 행과 열로 이루어진 테이블 형태의 데이터 구조로, 주로 데이터 분석 및 처리에 사용됩니다.\n예시: 판다스 라이브러리에서 DataFrame은 다양한 데이터 타입의 열을 가질 수 있으며, 데이터 조작과 분석을 용이하게 합니다.\n연관키워드: 데이터 분석, 판다스, 데이터 처리\n\nAttention 메커니즘\n\n정의: Attention 메커니즘은 딥러닝에서 중요한 정보에 더 많은 '주의'

### FAISS

In [ ]:
import faiss
from langchain_community.docstore.in_memory import InMemoryDocstore
from langchain_community.vectorstores import FAISS
dimension_size = len(embeddings.embed_query("dimension 확인용"))


# 1. 초기화 (벡터저장소 생성) - 인덱스와 문서 저장소를 직접 구성해야 함
index = faiss.IndexFlatL2(dimension_size)
db = FAISS(embedding_function=embeddings, index=index, docstore=InMemoryDocstore(), index_to_docstore_id={})
db.add_documents(split_doc1 + split_doc2)

['dc5ed93d-63a3-4314-b552-5c79804fbc9f',
 '850b43b4-315f-45aa-ac16-d88e5fdd7d52',
 'ad46788d-da6d-40d6-8878-ed960cb0972c',
 '84d6b6d7-85bb-4969-a423-6d729f9d589d',
 'b83b78cb-d820-4501-87d1-470ea24d850b',
 'f9b1da48-9a91-4af3-b52f-0e7bb014ba3e',
 'c48cb6e2-5589-4023-8b13-298eb65ebb51',
 'b2d77942-d8f0-44ca-ab9d-81b181133388',
 '65584862-63c5-4adb-b721-086d4fcc8f1e',
 'da00425f-2f17-4e9e-8e0a-bd27a087b19d',
 '0d728506-a469-40d4-bb48-82c31994c666',
 '38b2f3f2-47eb-4e8c-9127-21af449a0ea3',
 '41f95446-30ad-477c-8ff7-561ed732ac77',
 '66e4436a-f501-43a9-ab27-f2657e0c44d5',
 '75a8160d-1a5e-462d-bd2a-2022012a752a',
 'de442334-a57f-47a2-8dfa-30775570b11a',
 '0a271803-2c4f-4a48-8157-cf10b5b1cc56']

In [ ]:
# 2. 유사도 검색 (점수 포함)
query = "딥러닝에서 사용하는 핵심 개념은?"
results = db.similarity_search_with_score(query, k=2)
for doc, score in results:
    print(f"[거리 {score:.4f}] {doc.page_content[:50]}...")

[거리 0.2713] 정의: TF-IDF는 문서 내에서 단어의 중요도를 평가하는 데 사용되는 통계적 척도입니다....
[거리 0.2815] 정의: DataFrame은 행과 열로 이루어진 테이블 형태의 데이터 구조로, 주로 데이터 ...


In [ ]:
# 3. 텍스트로부터 추가
db.add_texts(["LangChain은 LLM 애플리케이션 개발을 돕는 프레임워크입니다."])

# 4. 문서(Document)로부터 추가
db.add_documents([Document(page_content="RAG는 검색 증강 생성 기법입니다.", metadata={"source": "추가문서"})])

['019eee20-25c4-4bd9-9955-73855a9fb6cc']

In [ ]:
# 5. 문서 삭제 - 내부 docstore를 직접 조회해서 id를 찾아야 함
target_ids = [k for k, v in db.docstore._dict.items() if v.metadata.get("source") == "추가문서"]
db.delete(target_ids)

True

In [ ]:
# 6. 저장 - 명시적으로 호출해야 함
db.save_local("./faiss_db")

# 7. 로드 - pickle 역직렬화 경고로 인해 allow_dangerous_deserialization=True 필요
db_loaded = FAISS.load_local("./faiss_db", embeddings, allow_dangerous_deserialization=True)

In [ ]:
# 8. 저장된 데이터 확인 - 내부 구조를 직접 조회해야 함
print(f"저장된 청크 수: {len(db_loaded.index_to_docstore_id)}")
first_id = list(db_loaded.index_to_docstore_id.values())[0]
print(db_loaded.docstore._dict[first_id].page_content[:50])

# 9. 검색기로 변환
retriever = db_loaded.as_retriever(search_type="similarity", search_kwargs={"k": 2})
retriever.invoke(query)

저장된 청크 수: 18
Semantic Search

정의: 의미론적 검색은 사용자의 질의를 단순한 키워드 매칭을


[Document(id='c48cb6e2-5589-4023-8b13-298eb65ebb51', metadata={'source': 'data/nlp-keywords.txt'}, page_content='정의: TF-IDF는 문서 내에서 단어의 중요도를 평가하는 데 사용되는 통계적 척도입니다. 이는 문서 내 단어의 빈도와 전체 문서 집합에서 그 단어의 희소성을 고려합니다.\n예시: 많은 문서에서 자주 등장하지 않는 단어는 높은 TF-IDF 값을 가집니다.\n연관키워드: 자연어 처리, 정보 검색, 데이터 마이닝\n\nDeep Learning\n\n정의: 딥러닝은 인공신경망을 이용하여 복잡한 문제를 해결하는 머신러닝의 한 분야입니다. 이는 데이터에서 고수준의 표현을 학습하는 데 중점을 둡니다.\n예시: 이미지 인식, 음성 인식, 자연어 처리 등에서 딥러닝 모델이 활용됩니다.\n연관키워드: 인공신경망, 머신러닝, 데이터 분석\n\nSchema\n\n정의: 스키마는 데이터베이스나 파일의 구조를 정의하는 것으로, 데이터가 어떻게 저장되고 조직되는지에 대한 청사진을 제공합니다.\n예시: 관계형 데이터베이스의 테이블 스키마는 열 이름, 데이터 타입, 키 제약 조건 등을 정의합니다.\n연관키워드: 데이터베이스, 데이터 모델링, 데이터 관리\n\nDataFrame'),
 Document(id='b2d77942-d8f0-44ca-ab9d-81b181133388', metadata={'source': 'data/nlp-keywords.txt'}, page_content="정의: DataFrame은 행과 열로 이루어진 테이블 형태의 데이터 구조로, 주로 데이터 분석 및 처리에 사용됩니다.\n예시: 판다스 라이브러리에서 DataFrame은 다양한 데이터 타입의 열을 가질 수 있으며, 데이터 조작과 분석을 용이하게 합니다.\n연관키워드: 데이터 분석, 판다스, 데이터 처리\n\nAttention 메커니즘\n\n정의: Attention 메커니즘은 딥러닝에서 중요한 정보에 더 많은 '주의'

# Retriever

In [ ]:
from langchain_community.retrievers import BM25Retriever
from langchain_classic.retrievers.ensemble import EnsembleRetriever

query = "딥러닝에서 사용하는 핵심 개념은?"
k = 3

In [ ]:
# 1. Dense
dense_results = db.similarity_search_with_score(query, k=k)
dense_retriever = db.as_retriever(search_kwargs={"k": k})

# 2. Sparse (BM25) 생성
bm25_retriever = BM25Retriever.from_documents(split_doc1 + split_doc2)
bm25_retriever.k = k
sparse_results = bm25_retriever.invoke(query)

# 3. Ensemble로 합치기
ensemble_retriever = EnsembleRetriever(
    retrievers=[bm25_retriever, dense_retriever],
    weights=[0.5, 0.5],) # [sparse 비중, dense 비중]
ensemble_results = ensemble_retriever.invoke(query)

In [ ]:
# 4. 세 결과 출력
print("=== Dense (의미 기반) ===")
for doc, score in dense_results:
    print(f"[거리 {score:.4f}] {doc.page_content[:50]}...")

print("\n=== Sparse (키워드 기반) ===")
for doc in sparse_results:
    print(f"{doc.page_content[:50]}...")

print("\n=== Hybrid (Ensemble) ===")
for doc in ensemble_results:
    print(f"{doc.page_content[:50]}...")

=== Dense (의미 기반) ===
[거리 0.2713] 정의: TF-IDF는 문서 내에서 단어의 중요도를 평가하는 데 사용되는 통계적 척도입니다....
[거리 0.2815] 정의: DataFrame은 행과 열로 이루어진 테이블 형태의 데이터 구조로, 주로 데이터 ...
[거리 0.3039] 정의: HuggingFace는 자연어 처리를 위한 다양한 사전 훈련된 모델과 도구를 제공하...

=== Sparse (키워드 기반) ===
정의: DataFrame은 행과 열로 이루어진 테이블 형태의 데이터 구조로, 주로 데이터 ...
정의: ESG는 기업의 환경, 사회, 지배구조 측면을 고려하는 투자 접근 방식입니다.
예시...
정의: 주식 리서치는 기업의 재무 상태, 사업 모델, 경쟁력 등을 분석하여 투자 의사 결정...

=== Hybrid (Ensemble) ===
정의: DataFrame은 행과 열로 이루어진 테이블 형태의 데이터 구조로, 주로 데이터 ...
정의: TF-IDF는 문서 내에서 단어의 중요도를 평가하는 데 사용되는 통계적 척도입니다....
정의: ESG는 기업의 환경, 사회, 지배구조 측면을 고려하는 투자 접근 방식입니다.
예시...
정의: 주식 리서치는 기업의 재무 상태, 사업 모델, 경쟁력 등을 분석하여 투자 의사 결정...
정의: HuggingFace는 자연어 처리를 위한 다양한 사전 훈련된 모델과 도구를 제공하...


In [ ]:
# 5. Dense·Sparse·Ensemble 문서 집합 비교

# Ensemble 결과도 동일하게 top-k로 자르기
ensemble_topk = ensemble_results[:k]

dense_texts = {doc.page_content[:30] for doc, score in dense_results}
sparse_texts = {doc.page_content[:30] for doc in sparse_results}
ensemble_texts = {doc.page_content[:30] for doc in ensemble_topk}

print(f"=== Dense ∩ Sparse (각 top-{k}): {len(dense_texts & sparse_texts)}개 ===")
print(f"=== Dense ∩ Ensemble top-{k}: {len(dense_texts & ensemble_texts)}개 / {k}개 중 ===")
print(f"=== Sparse ∩ Ensemble top-{k}: {len(sparse_texts & ensemble_texts)}개 / {k}개 중 ===")

=== Dense ∩ Sparse (각 top-3): 1개 ===
=== Dense ∩ Ensemble top-3: 2개 / 3개 중 ===
=== Sparse ∩ Ensemble top-3: 2개 / 3개 중 ===
